# Full MEDIC labeling, feature caching and classifier study
Readable orchestration notebook. Build in progress: not submitted or executed on Kaggle. No Base64 source. Shared project source must be attached as a versioned managed input. Labels are VLM pseudo-labels, not verified urgency. Training requires a completed verified labeling checkpoint.

## Round0 — environment and source identity

In [ ]:
print('ROUND0 START',flush=True)
import sys,json,time,importlib.metadata
from pathlib import Path
import torch
PROJECT=Path('/kaggle/input/semantic-scheduler-source')
if not (PROJECT/'experiments/full_medic_labeling/campaign.py').is_file():
    raise RuntimeError('Attach verified readable project source input before execution')
sys.path.insert(0,str(PROJECT))
CONFIG=json.loads((PROJECT/'experiments/full_medic_labeling/config.json').read_text())
print(sys.executable,sys.version,'GPU count',torch.cuda.device_count())
print({p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','scikit-learn','pyarrow']})
assert torch.cuda.device_count()==2
RUN=Path('/kaggle/working/experiments/full_medic_labeling/runs')/time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())
RUN.mkdir(parents=True)
print('Outputs',RUN,'ROUND0 END',flush=True)


## Round1 — pinned dataset inventory
Download/cache implementation is shard-based; saving all source shards requires the dataset-storage stage to be completed before this notebook is launched.

In [ ]:
print('ROUND1 START',flush=True)
from experiments.full_medic_labeling.campaign import inventory
from experiments.spi_classifier_repair.resume_store import atomic_json
source=inventory();atomic_json(RUN/'dataset_inventory.json',source)
print('Rows',source['splits'],'Bytes',sum(x['bytes'] for x in source['shards']),flush=True)
print('ROUND1 END',flush=True)


## Round2 — system prompt
The rubric is the system message, never annotation-derived input. GPU1 is reserved for the VLM; GPU0 sequentially extracts the four encoders and YOLO. Device orchestration is pending integration; do not substitute the older two-VLM-replica loop.

In [ ]:
print('ROUND2 PROMPT START',flush=True)
from experiments.full_medic_labeling.teacher import SYSTEM_PROMPT,PROMPT_SHA256
print(SYSTEM_PROMPT)
print('Prompt SHA256',PROMPT_SHA256,flush=True)
(RUN/'SYSTEM_PROMPT.txt').write_text(SYSTEM_PROMPT,encoding='utf8')
print('ROUND2 PROMPT END',flush=True)


## Round3 — labeling and feature caching
This cell deliberately blocks while the revised single-VLM/all-encoder pipeline is unfinished. No false completion or silently different GPU layout.

In [ ]:
raise RuntimeError('BUILD IN PROGRESS: integrate GPU1 teacher / GPU0 all-encoder cache and saved dataset stage before submission')

## Round break — saved labels and balancing
Verify71198 source-row outcomes and checkpoint hash. Preserve unavailable labels. Freeze duplicate-grouped roles. Resample only isolated fit folds; validation/test remain natural. Do not force classes through pseudo-label generation.

In [ ]:
print('LABEL COMPLETION GATE START',flush=True)
from experiments.full_medic_labeling.training import require_complete
connection,receipt=require_complete(RUN/'labels');connection.close()
print(receipt['progress'],'LABEL COMPLETION GATE END',flush=True)


## Rounds4–5 — symmetric classifier matrix
32 ML plus16 neural pairs. Balancing dispatch and separated round execution remain pending. The old48-pair train_all helper is not yet the final balancing runner.

In [ ]:
print('PLANNED MATRIX',CONFIG['encoders'],CONFIG['ml_heads'],CONFIG['neural_heads'])
print('Unique pairs',CONFIG['unique_pairs'],'Balancing',CONFIG['balancing'])
print('Do not report unexecuted costs or model scores.')


## Costs and persistence
Separate teacher, frozen extraction, fitting/search and deployment measurements. CPU/GPU each use fixed10development images,10warmups,30CPU/100GPU calls; synchronize CUDA and separate memory passes.2FLOPs/MAC; unsupported operators remain partial. Relevance/KB/serialization are not executed in this offline study. Every computational stage must have start/end, elapsed time, incremental logs and checksum-verified saved outputs before launch.